In [1]:
# We're rebuilding the foundation from Agents 1 and 2 in one cell
# In a real system these would be separate services running independently
# For now we consolidate them here so the notebook is self-contained

import os
import anthropic
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

load_dotenv()

# ── Vector database (Agent 2) ──
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="bank_products",
    embedding_function=embedding_fn
)

# Load the 5 products into the database
products = [
    {"id": "PROD_001", "name": "Voyager Premium Travel Card",
     "description": """Premium credit card designed for frequent international travellers. 
     Includes unlimited airport lounge access worldwide, zero foreign transaction fees, 
     travel insurance, and 3x reward points on flights and hotels. 
     Annual fee: €150. Requires minimum income of €35,000."""},
    {"id": "PROD_002", "name": "GlobalCash FX Account",
     "description": """Multi-currency account that holds and converts between 30 currencies 
     at interbank rates with no markup. Ideal for customers who regularly send money abroad 
     or make international purchases. No monthly fee. Includes a linked debit card."""},
    {"id": "PROD_003", "name": "HomeSaver Account",
     "description": """High-interest savings account for customers saving towards a home purchase. 
     Offers 4.5% AER on balances up to €50,000. Requires minimum monthly deposit of €200. 
     Bonus interest rate if no withdrawals made in 12 months."""},
    {"id": "PROD_004", "name": "FlexiLoan Personal Loan",
     "description": """Unsecured personal loan from €1,000 to €50,000 with flexible repayment 
     terms of 1 to 7 years. Competitive fixed interest rate. Suitable for home improvements, 
     car purchase, or consolidating existing debts. Fast online approval."""},
    {"id": "PROD_005", "name": "FamilyFirst Current Account",
     "description": """Family banking package that includes accounts for parents and up to 3 children. 
     Includes pocket money features, spending controls for children, joint account option, 
     and family cashback on supermarket and school-related spending. Monthly fee: €8."""}
]

collection.add(
    ids=[p["id"] for p in products],
    documents=[p["description"] for p in products],
    metadatas=[{"name": p["name"], "id": p["id"]} for p in products]
)

# ── Claude client (Agent 1) ──
claude = anthropic.Anthropic()

print("✅ All systems ready — Agents 1 and 2 foundations loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ All systems ready — Agents 1 and 2 foundations loaded


In [2]:
# In a real system, this would come directly from Agent 1 running automatically
# For now we paste in the profile Agent 1 already produced for Sarah
# This is called "mocking the upstream output" — standard practice when building pipelines

customer_profile = """
CUSTOMER INTELLIGENCE PROFILE
==============================
Customer ID: CUST_001
Name: Sarah Mitchell

BEHAVIOURAL AFFINITIES:
- Travel Affinity: HIGH — Four international trips booked across three continents within six months
- FX/International Need: HIGH — Multiple currency exchanges across USD, GBP, and JPY
- Premium Product Fit: HIGH — Travel frequency and ancillary costs indicate premium product alignment
- Savings Propensity: LOW — No savings account despite consistent income patterns

TOP IDENTIFIED NEED:
Sarah requires a travel-optimised banking product that consolidates FX management and provides 
international convenience benefits.

CURRENT PRODUCTS:
- Basic current account
- Standard debit card (no travel benefits)
- No credit card
- No savings account
"""

print("✅ Customer profile loaded")
print(customer_profile)


✅ Customer profile loaded

CUSTOMER INTELLIGENCE PROFILE
Customer ID: CUST_001
Name: Sarah Mitchell

BEHAVIOURAL AFFINITIES:
- Travel Affinity: HIGH — Four international trips booked across three continents within six months
- FX/International Need: HIGH — Multiple currency exchanges across USD, GBP, and JPY
- Premium Product Fit: HIGH — Travel frequency and ancillary costs indicate premium product alignment
- Savings Propensity: LOW — No savings account despite consistent income patterns

TOP IDENTIFIED NEED:
Sarah requires a travel-optimised banking product that consolidates FX management and provides 
international convenience benefits.

CURRENT PRODUCTS:
- Basic current account
- Standard debit card (no travel benefits)
- No credit card
- No savings account



In [3]:
# We extract the top customer need and use it to query the vector database
# This is Agent 2 being called by Agent 3 — agents working together

search_query = "frequent international traveller needing FX management and travel benefits"

matched_products = collection.query(
    query_texts=[search_query],
    n_results=2
)

# Format the results cleanly for the next step
product_context = ""
for name, desc in zip(
    [m["name"] for m in matched_products["metadatas"][0]],
    matched_products["documents"][0]
):
    product_context += f"\nPRODUCT: {name}\n{desc}\n"

print("✅ Matched products retrieved from vector database:")
print(product_context)

✅ Matched products retrieved from vector database:

PRODUCT: GlobalCash FX Account
Multi-currency account that holds and converts between 30 currencies 
     at interbank rates with no markup. Ideal for customers who regularly send money abroad 
     or make international purchases. No monthly fee. Includes a linked debit card.

PRODUCT: Voyager Premium Travel Card
Premium credit card designed for frequent international travellers. 
     Includes unlimited airport lounge access worldwide, zero foreign transaction fees, 
     travel insurance, and 3x reward points on flights and hotels. 
     Annual fee: €150. Requires minimum income of €35,000.



In [4]:
# This is the heart of Agent 3
# We send BOTH the customer profile AND the matched products to Claude
# Claude's job is to reason about WHY this product fits THIS customer
# and produce a structured opportunity card

prompt = f"""
You are an Opportunity Discovery Agent for a bank.

You have been given:
1. A customer intelligence profile
2. A list of potentially relevant bank products

Your job is to identify the single best opportunity and explain clearly why 
this product is relevant to this specific customer.

IMPORTANT RULES:
- Do not reference specific transaction amounts or dates
- Focus on behavioural patterns, not raw data
- Be concise — the executive needs to act on this quickly
- Flag if the timing might not be right

CUSTOMER PROFILE:
{customer_profile}

POTENTIALLY RELEVANT PRODUCTS:
{product_context}

Produce your output in exactly this format:

OPPORTUNITY CARD
================
Customer: [name] ([id])

RECOMMENDED PRODUCT: [product name]

WHY THIS FITS THIS CUSTOMER:
[2-3 sentences explaining the match based on behavioural patterns only]

ESTIMATED VALUE TO CUSTOMER:
[One sentence on what the customer gains — no specific figures, keep it qualitative]

CONVERSATION READINESS: [YES / MAYBE / NO]
REASON: [One sentence on whether now is a good moment to raise this]

SUGGESTED OPENING LINE FOR EXECUTIVE:
"[The exact words the executive could say]"
"""

response = claude.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)

print(response.content[0].text)

OPPORTUNITY CARD
Customer: Sarah Mitchell (CUST_001)

RECOMMENDED PRODUCT: Voyager Premium Travel Card

WHY THIS FITS THIS CUSTOMER:
Sarah's travel pattern demonstrates consistent, frequent international activity across multiple continents with regular FX needs—a clear indicator of premium travel card alignment. The unlimited lounge access, zero foreign transaction fees, and travel insurance directly address the friction points inherent in her current travel behaviour. Her income profile supports the premium tier positioning.

ESTIMATED VALUE TO CUSTOMER:
Elimination of foreign transaction costs across her travel spending, plus ancillary travel conveniences (lounge access, travel insurance) that reduce friction on her frequent international journeys.

CONVERSATION READINESS: YES
REASON: Active travel behaviour is happening now, making the timing optimal to introduce benefits that address her current pain points.

SUGGESTED OPENING LINE FOR EXECUTIVE:
"Sarah, I've noticed you're travell